# Redispatch sanity check — grid-operator lens
Source: `data/raw/shn_operations_last_2y/*.parquet` (Schleswig-Holstein Netz DSO, north Germany — wind-heavy region).
Window: **2024-01-01 → 2026-04-01** (capped to external-data coverage).

Each raw row is one **asset** (transformer/feeder) inside a redispatch operation. Multiple assets share an `operationId` and a constrained transformer (`locationBottleneck`, e.g. `…-T122`).
One **op** = unique `(start, end, location)` tuple — that's the correct unit for the time series.

Reason codes (grid-analyst reading):
- `Netzengpass I/U` — congestion on the internal grid (primary driver).
- `TenneT2` — upstream TSO-triggered curtailment.
- `Sonstiges` / `Sonstige` — 'other'.
- `Funktionsnachweis` — functional test (not a real congestion event; candidate to drop).
- `Spitzenkappung` — peak-capping (EEG §11.2).
- `Abschaltung Erzeugung` — generation shutdown.

`liabilityOfCompensation`: **EEG** = renewable feed-in curtailment (DSO pays producer); **EnWG** = conventional redispatch.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

OPS_DIR      = ROOT / 'data' / 'raw' / 'shn_operations_last_2y'
EXTERNAL_PQ  = ROOT / 'data' / 'external' / 'smard_weather_2024_to_mar2026.parquet'

START_CUTOFF = pd.Timestamp('2024-01-01')
END_CUTOFF   = pd.Timestamp('2026-04-01')   # match external-data upper bound
WEEK_MIN     = 7 * 24 * 60

print(f'ops dir  : {OPS_DIR}  (exists: {OPS_DIR.exists()})')
print(f'external : {EXTERNAL_PQ}  (exists: {EXTERNAL_PQ.exists()})')
print(f'window   : {START_CUTOFF.date()}  →  {END_CUTOFF.date()}')

## 1. Load raw

In [ ]:
files = sorted(OPS_DIR.glob('chunk_*.parquet'))
raw = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)
raw['start'] = pd.to_datetime(raw['start'])
raw['end']   = pd.to_datetime(raw['end'])
print(f'raw asset-level rows: {len(raw):,}')
print(f'raw date range     : {raw["start"].min()}  →  {raw["start"].max()}')

## 2. Cleaning pipeline
Applied as a single auditable sequence. Each step reports rows in / rows out.

In [ ]:
def step(name, before, after):
    print(f'  {name:<42}  {before:>8,}  →  {after:>8,}   (-{before-after:,})')

df = raw.copy()
n0 = len(df)
print(f'start: {n0:,} asset-level rows\n')

# (a) window
b = len(df); df = df[(df['start'] >= START_CUTOFF) & (df['start'] < END_CUTOFF)].reset_index(drop=True)
step('window 2024-01-01 → 2026-04-01', b, len(df))

# (b) drop dead columns
df = df.drop(columns=['srId', 'Reason'])   # srId 100% NaN, Reason duplicates reason
print('  drop srId (all NaN) and Reason (dup)')

# (c) drop 'UW' placeholder
b = len(df); df = df[df['location'] != 'UW'].reset_index(drop=True)
step("drop location=='UW' placeholder", b, len(df))

# (d) collapse asset-level → op-level
b = len(df); df_ops = df.drop_duplicates(subset=['start', 'end', 'location']).reset_index(drop=True)
step('dedup to unique ops (start,end,location)', b, len(df_ops))

# (e) duration_calc, drop sentinel / negative / >1 week
df_ops['duration_min'] = (df_ops['end'] - df_ops['start']).dt.total_seconds() / 60
b = len(df_ops); df_ops = df_ops[(df_ops['duration_min'] > 0) & (df_ops['duration_min'] <= WEEK_MIN)].reset_index(drop=True)
step('drop bad duration (<=0 or >1 week)', b, len(df_ops))

# (f) normalize town; explode multi-town rows
df_ops['town_raw'] = df_ops['location'].str.replace(r'^UW\s+', '', regex=True).str.strip()
df_ops['town_list'] = df_ops['town_raw'].str.split(',').apply(
    lambda xs: [t.replace('UW ', '').strip() for t in xs]
)
df_town = df_ops.explode('town_list').rename(columns={'town_list': 'town'}).reset_index(drop=True)
print(f'\nop-level rows           : {len(df_ops):,}')
print(f'after exploding multi-town: {len(df_town):,}  (fan-out factor {len(df_town)/len(df_ops):.3f})')
print(f'unique towns              : {df_town["town"].nunique()}')

## 3. Where is the grid hurting? — towns & transformers

In [ ]:
# ops per town + total constrained minutes per town (intensity)
town_stats = (df_town.groupby('town')
              .agg(ops=('start', 'size'),
                   total_min=('duration_min', 'sum'),
                   median_min=('duration_min', 'median'))
              .sort_values('total_min', ascending=False))
town_stats['total_hours'] = town_stats['total_min'] / 60
print('Top 15 towns by total constrained hours:')
town_stats.head(15)

In [ ]:
# Top constrained transformers (locationBottleneck identifies the transformer, e.g. '...-T122')
bn = (df_ops.groupby('locationBottleneck')
      .agg(ops=('start', 'size'),
           total_hours=('duration_min', lambda s: s.sum()/60),
           example_location=('location', 'first'))
      .sort_values('total_hours', ascending=False))
print(f'unique bottleneck transformers: {len(bn):,}')
print('\nTop 15 by constrained hours:')
bn.head(15)

## 4. Duration, concurrency & dispatch-desk load

In [ ]:
print('duration (min) — clean ops:')
print(df_ops['duration_min'].describe(percentiles=[.5, .9, .99]).round(1))

fig, ax = plt.subplots(figsize=(9, 3))
df_ops['duration_min'].clip(upper=df_ops['duration_min'].quantile(0.99)).hist(bins=60, ax=ax)
ax.set_xlabel('duration (min, clipped at p99)')
ax.set_title('op duration distribution')
plt.show()

In [ ]:
# concurrent ops — sweep-line on start/end events
events = pd.concat([
    df_ops[['start']].rename(columns={'start': 't'}).assign(delta=1),
    df_ops[['end']].rename(columns={'end': 't'}).assign(delta=-1),
]).sort_values('t').reset_index(drop=True)
events['concurrent'] = events['delta'].cumsum()

print('concurrent ops — distribution:')
print(events['concurrent'].describe(percentiles=[.5, .9, .99]).round(1))

# hourly mean concurrency
conc_h = (events.set_index('t')['concurrent']
          .resample('1h').mean().dropna())
fig, ax = plt.subplots(figsize=(13, 3.5))
conc_h.rolling(24*7).mean().plot(ax=ax)
ax.set_title('mean concurrent ops (7-day rolling)')
ax.set_ylabel('ops active')
plt.show()

## 5. Reason × liability — what drives redispatch here?
`Netzengpass*` + `EEG` = renewable (wind) curtailment for internal grid congestion — the SHN signature.

In [ ]:
ct = pd.crosstab(df_ops['reason'], df_ops['liabilityOfCompensation'], margins=True, margins_name='total')
ct_sorted = ct.sort_values('total', ascending=False)
print('reason × liability (op count):')
print(ct_sorted)

print('\ncontrol stage × liability (op count):')
print(pd.crosstab(df_ops['controlStage'], df_ops['liabilityOfCompensation']))

## 6. Temporal patterns — hour × weekday heatmap
Wind-driven curtailments should spike overnight (low load, high wind). Weekend vs weekday shows demand vs wind dominance.

In [ ]:
df_ops['hour'] = df_ops['start'].dt.hour
df_ops['dow']  = df_ops['start'].dt.day_name()
dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
heat = (df_ops.groupby(['dow', 'hour']).size()
        .unstack('hour').reindex(dow_order).fillna(0))

fig, ax = plt.subplots(figsize=(12, 3.2))
im = ax.imshow(heat.values, aspect='auto', cmap='viridis')
ax.set_xticks(range(24)); ax.set_xticklabels(range(24))
ax.set_yticks(range(7));  ax.set_yticklabels(dow_order)
ax.set_xlabel('hour of day'); ax.set_title('op starts — hour × weekday')
plt.colorbar(im, ax=ax, label='op count')
plt.show()

## 7. Load external cache & sanity-link to redispatch
Hourly SMARD + weather. Quick check: does high wind-onshore generation coincide with more ops?

In [ ]:
ext = pd.read_parquet(EXTERNAL_PQ)
ext['timestamp'] = pd.to_datetime(ext['timestamp'], utc=True).dt.tz_localize(None)
print(f'external rows : {len(ext):,}')
print(f'range         : {ext["timestamp"].min()}  →  {ext["timestamp"].max()}')
wanted = ['residual_load', 'wind_onshore', 'wind_offshore', 'solar', 'day_ahead',
          'wx_wind_speed_100m', 'wx_shortwave_radiation', 'wx_temperature_2m', 'wx_cloud_cover']
print('\nNaN in model-ready columns:')
print(ext[wanted].isna().sum())

In [ ]:
# hourly op count vs wind_onshore
ops_hourly = (df_ops.assign(hr=lambda d: d['start'].dt.floor('h'))
              .groupby('hr').size().rename('ops'))
merged = (ext.set_index('timestamp')[['wind_onshore', 'residual_load']]
          .join(ops_hourly, how='inner'))
merged['ops'] = merged['ops'].fillna(0)

corr = merged.corr().loc['ops']
print('Pearson correlation of hourly op count with:')
print(corr.drop('ops').round(3))

# quintile plot — mean hourly ops by wind quintile
merged['wind_q'] = pd.qcut(merged['wind_onshore'], 5, labels=['Q1 low','Q2','Q3','Q4','Q5 high'])
q = merged.groupby('wind_q', observed=True)['ops'].mean()
fig, ax = plt.subplots(figsize=(7, 3))
q.plot(kind='bar', ax=ax)
ax.set_ylabel('mean ops started per hour')
ax.set_title('ops vs DE-LU wind-onshore generation quintile')
plt.show()

## 8. Summary counters + decision gate

In [ ]:
print('=== clean dataset ready ===')
print(f'asset rows (raw, in window)   : {((raw["start"] >= START_CUTOFF) & (raw["start"] < END_CUTOFF)).sum():,}')
print(f'op-level rows                 : {len(df_ops):,}')
print(f'town-exploded rows            : {len(df_town):,}')
print(f'unique locations              : {df_ops["location"].nunique()}')
print(f'unique towns                  : {df_town["town"].nunique()}')
print(f'unique bottleneck transformers: {df_ops["locationBottleneck"].nunique()}')
print(f'ops date range                : {df_ops["start"].min()}  →  {df_ops["start"].max()}')
print('\nNext: src/build_timeseries.py — range-join df_town to 15-min grid × town list → ts_15min.parquet')

In [ ]:
import pandas as pd
df = pd.read_parquet(r'C:\Users\ashis\OneDrive\Desktop\Restart\Projects\Redispatch\data\processed\ts_15min.parquet')
df.set_index('ts', inplace=True)
df.head()

In [ ]:
df['active_ops'].groupby(df['town']).plot()

In [ ]:
wide = pd.read_parquet(r'C:\Users\ashis\OneDrive\Desktop\Restart\Projects\Redispatch\data\processed\ts_15min_wide.parquet')
wide.loc['2025-04-11 08:00:00']           # who was active at the busiest slot
wide['Husum'].plot()                       # one-town time series
(wide > 0).sum().sort_values()             # active-slot count per town

In [ ]:
import time, re, pandas as pd, requests
from pathlib import Path

ROOT = Path(r'C:\Users\ashis\OneDrive\Desktop\Restart\Projects\Redispatch')
OUT  = ROOT / 'data' / 'external' / 'towns_geo.parquet'
UA   = {'User-Agent': 'redispatch-research/0.1 (connectashish28@gmail.com)'}

geo  = pd.read_parquet(OUT)
miss = geo[geo['lat'].isna()].copy()
print(f'{len(miss)} still unresolved')

KV_PRIV   = re.compile(r'^\s*\d+\s*kV\s+', flags=re.IGNORECASE)
PRIV_TAG  = re.compile(r'\s*\((privat|public)\)\s*$', flags=re.IGNORECASE)
QUAL      = re.compile(
    r'\s+(West|Ost|Nord|S(ü|u)d|Mitte|Neu|Alt|Wind|Adlerhorst|I{1,3}|\d+)$',
    flags=re.IGNORECASE,
)

def clean(name: str) -> str:
    s = KV_PRIV.sub('', name)            # strip leading "110kV "
    s = PRIV_TAG.sub('', s)              # strip trailing "(Privat)"
    s = re.sub(r'[-]', ' ', s)           # "Heiligenhafen-Ost" -> "Heiligenhafen Ost"
    return s.strip()

def variants(name: str):
    c = clean(name)
    yield c
    bare = QUAL.sub('', c).strip()
    if bare and bare != c: yield bare
    if ' ' in bare: yield bare.split()[0]    # last resort: first word

def query(q, region=True):
    params = {'q': q + (', Schleswig-Holstein, Germany' if region else ', Germany'),
              'format': 'json', 'limit': 1, 'countrycodes': 'de',
              'featuretype': 'settlement'}
    try:
        r = requests.get('https://nominatim.openstreetmap.org/search',
                         params=params, headers=UA, timeout=15)
        j = r.json()
        return (float(j[0]['lat']), float(j[0]['lon'])) if j else (None, None)
    except Exception:
        return (None, None)

for i, town in enumerate(miss['town'].tolist(), 1):
    found, used = None, None
    for v in dict.fromkeys(variants(town)):  # de-dup, preserve order
        for region in (True, False):
            lat, lon = query(v, region=region)
            time.sleep(1.1)
            if lat is not None:
                found, used = (lat, lon), f'{v} (SH={region})'
                break
        if found: break
    if found:
        geo.loc[geo['town'] == town, ['lat','lon']] = list(found)
        print(f'  OK   {town:45s} via "{used}"')
    else:
        print(f'  --   {town}')
    if i % 10 == 0:
        geo.to_parquet(OUT, index=False)

geo.to_parquet(OUT, index=False)
hit = geo['lat'].notna().sum()
print(f'\nfinal: {hit}/{len(geo)} ({hit/len(geo):.0%})')
print('still missing:', geo.loc[geo['lat'].isna(), 'town'].tolist())

In [ ]:
import pandas as pd
from pathlib import Path

ROOT = Path(r'C:\Users\ashis\OneDrive\Desktop\Restart\Projects\Redispatch')
OUT  = ROOT / 'data' / 'external' / 'towns_geo.parquet'

geo = pd.read_parquet(OUT)

# 1. Find the real string behind 'Str?bbel' — '?' is just a console glyph
mask = geo['lat'].isna()
real_names = geo.loc[mask, 'town'].tolist()
print('actual missing strings:', [repr(n) for n in real_names])
# Expect to see: 'Strübbel' (Dithmarschen) and 'Gravelund' (Nordfriesland)

# 2. Apply manual overrides
overrides = {
    'Strübbel':  (54.3092, 8.9082),   # Strübbel, Dithmarschen (~5km W of Heide)
    'Gravelund': (54.8270, 9.0460),   # Gravelund hamlet, near Klixbüll/Niebüll
}
for name, (lat, lon) in overrides.items():
    geo.loc[geo['town'] == name, ['lat','lon']] = [lat, lon]

# 3. If the actual string is mojibake (e.g. 'Str\xfcbbel' shown as 'Str?bbel'),
#    handle by repr match too — just in case
for nm in real_names:
    if 'bbel' in nm and nm not in overrides:
        geo.loc[geo['town'] == nm, ['lat','lon']] = [54.3092, 8.9082]
        print(f'patched mojibake "{nm}" -> Strübbel coords')

geo.to_parquet(OUT, index=False)
hit = geo['lat'].notna().sum()
print(f'final: {hit}/{len(geo)} ({hit/len(geo):.0%})')
print('still missing:', geo.loc[geo['lat'].isna(), 'town'].tolist())

In [ ]:
import pandas as pd, folium
from pathlib import Path

ROOT = Path(r'C:\Users\ashis\OneDrive\Desktop\Restart\Projects\Redispatch')
geo  = pd.read_parquet(ROOT / 'data/external/towns_geo.parquet')
wide = pd.read_parquet(ROOT / 'data/processed/ts_15min_wide.parquet')

intensity = (wide > 0).sum().rename('active_slots').reset_index()
intensity.columns = ['town', 'active_slots']
df = geo.dropna(subset=['lat','lon']).merge(intensity, on='town')

m = folium.Map(location=[54.3, 9.7], zoom_start=8, tiles='cartodbpositron')
maxv = df['active_slots'].max()
for _, r in df.iterrows():
    folium.CircleMarker(
        location=(r.lat, r.lon),
        radius=3 + 20 * (r.active_slots / maxv),
        popup=f"{r.town}: {int(r.active_slots):,} active 15-min slots",
        color='crimson', fill=True, fill_opacity=0.6,
    ).add_to(m)
m.save(str(ROOT / 'redispatch_map.html'))
m   # display inline in notebook

In [ ]:
import pandas as pd
ext = pd.read_parquet(r'C:\Users\ashis\OneDrive\Desktop\Restart\Projects\Redispatch\data\external\smard_weather_2024_to_mar2026.parquet')
print(ext.shape)
print(ext.dtypes)
print(ext.head(3))
print('ts range:', ext.index.min() if ext.index.name=='ts' else ext.iloc[:,0].min(),
      '->',         ext.index.max() if ext.index.name=='ts' else ext.iloc[:,0].max())
